In [2]:



import pandas as pd
from pathlib import Path

print("pandas version:", pd.__version__)

# Locate the Raw Datasets
data_path = Path("../data/raw")

# Display the available csv files
csv_files = list(data_path.glob(".csv"))

print("Number of datasets:", len(csv_files))

for file in csv_files:
    print(file.name)



pandas version: 3.0.2
Number of datasets: 0


In [3]:

from pathlib import Path

print("Current working directory:", Path.cwd())

data_path = Path("../data/raw")

print("Looking for datasets in:", data_path.resolve())
print("Does the folder exist?", data_path.exists())


Current working directory: c:\Users\hp\Desktop\JOSEPHINE BASSEY FILES\E-Commerce-Sales-Analytics\Notebooks
Looking for datasets in: C:\Users\hp\Desktop\JOSEPHINE BASSEY FILES\E-Commerce-Sales-Analytics\Data\Raw
Does the folder exist? True


In [4]:

from pathlib import Path

data_path = Path("../Data/Raw")

print("Files inside the Raw folder:\n")

for file in data_path.iterdir():
    print(file.name)


Files inside the Raw folder:

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation(1).csv


In [5]:

import pandas as pd
from pathlib import Path

# Define the location of our raw datasets
data_path = Path("../Data/Raw")

# Find all CSV files
csv_files = sorted(data_path.glob("*.csv"))

print("Number of CSV files found:", len(csv_files))

# Load all CSV files into a dictionary
datasets = {}

for file in csv_files:
    dataset_name = file.stem
    datasets[dataset_name] = pd.read_csv(file)

    print(
        f"{dataset_name}: "
        f"{datasets[dataset_name].shape[0]:,} rows, "
        f"{datasets[dataset_name].shape[1]} columns"
    )


Number of CSV files found: 9
olist_customers_dataset: 99,441 rows, 5 columns
olist_geolocation_dataset: 1,000,163 rows, 5 columns
olist_order_items_dataset: 112,650 rows, 7 columns
olist_order_payments_dataset: 103,886 rows, 5 columns
olist_order_reviews_dataset: 99,224 rows, 7 columns
olist_orders_dataset: 99,441 rows, 8 columns
olist_products_dataset: 32,951 rows, 9 columns
olist_sellers_dataset: 3,095 rows, 4 columns
product_category_name_translation(1): 71 rows, 2 columns


In [6]:

# DATA QUALITY ASSESSMENT: MISSING VALUES

for name, df in datasets.items():

    print(f"\nDATASET: {name}")
    print("-" * 50)

    missing_count = df.isnull().sum()

    missing_percentage = (
        missing_count / len(df) * 100
    ).round(2)

    missing_report = pd.DataFrame({
        "Missing Values": missing_count,
        "Missing Percentage": missing_percentage
    })

    # Show only columns with missing values
    missing_report = missing_report[
        missing_report["Missing Values"] > 0
    ]

    if missing_report.empty:
        print("No missing values found.")
    else:
        print(missing_report)




DATASET: olist_customers_dataset
--------------------------------------------------
No missing values found.

DATASET: olist_geolocation_dataset
--------------------------------------------------
No missing values found.

DATASET: olist_order_items_dataset
--------------------------------------------------
No missing values found.

DATASET: olist_order_payments_dataset
--------------------------------------------------
No missing values found.

DATASET: olist_order_reviews_dataset
--------------------------------------------------
                        Missing Values  Missing Percentage
review_comment_title             87656               88.34
review_comment_message           58247               58.70

DATASET: olist_orders_dataset
--------------------------------------------------
                               Missing Values  Missing Percentage
order_approved_at                         160                0.16
order_delivered_carrier_date             1783                1.79
order

In [7]:

# Investigate orders with missing delivery dates

orders = datasets["olist_orders_dataset"]

missing_delivery = orders[
    orders["order_delivered_customer_date"].isna()
]

print("Orders with missing delivery dates:")
print(len(missing_delivery))

print("\nOrder status breakdown:")
print(
    missing_delivery["order_status"]
    .value_counts(dropna=False)
)


Orders with missing delivery dates:
2965

Order status breakdown:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


In [8]:

# Missing delivery dates by order status

delivery_status_report = (
    orders.groupby("order_status")
    ["order_delivered_customer_date"]
    .agg(
        total_orders="size",
        missing_delivery_dates=lambda x: x.isna().sum()
    )
)

delivery_status_report["missing_percentage"] = (
    delivery_status_report["missing_delivery_dates"]
    / delivery_status_report["total_orders"] * 100
).round(2)

print(delivery_status_report)


              total_orders  missing_delivery_dates  missing_percentage
order_status                                                          
approved                 2                       2              100.00
canceled               625                     619               99.04
created                  5                       5              100.00
delivered            96478                       8                0.01
invoiced               314                     314              100.00
processing             301                     301              100.00
shipped               1107                    1107              100.00
unavailable            609                     609              100.00


In [9]:

# Check delivered orders with missing delivery dates
delivered_missing = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
]

# Check cancelled orders with recorded delivery dates
cancelled_delivered = orders[
    (orders["order_status"] == "canceled") &
    (orders["order_delivered_customer_date"].notna())
]

print("Delivered orders missing delivery dates:", len(delivered_missing))
print("Cancelled orders with delivery dates:", len(cancelled_delivered))

# Show important columns
columns_to_check = [
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

display(delivered_missing[columns_to_check])
display(cancelled_delivered[columns_to_check])


Delivered orders missing delivery dates: 8
Cancelled orders with delivery dates: 6


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09 00:00:00
8791,dabf2b0e35b423f94618bf965fcb7514,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30 00:00:00
58266,770d331c84e5b214bd9dc70a10b829d0,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29 00:00:00
59332,8beb59392e21af5eb9547ae1a9938d06,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30 00:00:00
92636,65d1e226dfaeb8cdc42f665422522d14,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25 00:00:00
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08 00:00:00


In [10]:

# Create a working copy of the Orders dataset
orders_clean = orders.copy()

# Flag delivered orders without delivery dates
orders_clean["missing_delivered_date_flag"] = (
    (orders_clean["order_status"] == "delivered") &
    (orders_clean["order_delivered_customer_date"].isna())
)

# Flag cancelled orders with recorded delivery dates
orders_clean["cancelled_with_delivery_flag"] = (
    (orders_clean["order_status"] == "canceled") &
    (orders_clean["order_delivered_customer_date"].notna())
)

# Count flagged records
print("Delivered without date:",
      orders_clean["missing_delivered_date_flag"].sum())

print("Cancelled with delivery date:",
      orders_clean["cancelled_with_delivery_flag"].sum())

print("Original orders:", len(orders))
print("Working orders:", len(orders_clean))


Delivered without date: 8
Cancelled with delivery date: 6
Original orders: 99441
Working orders: 99441


In [11]:

# DATA QUALITY ASSESSMENT: DUPLICATES

duplicate_results = []

for name, df in datasets.items():

    total_rows = len(df)
    duplicate_rows = df.duplicated().sum()

    duplicate_percentage = (
        duplicate_rows / total_rows * 100
    ).round(2)

    duplicate_results.append({
        "Dataset": name,
        "Total Rows": total_rows,
        "Duplicate Rows": duplicate_rows,
        "Duplicate Percentage": duplicate_percentage
    })

duplicate_report = pd.DataFrame(duplicate_results)

display(duplicate_report)


,Dataset,Total Rows,Duplicate Rows,Duplicate Percentage
0,olist_customers_dataset,99441,0,0.00
1,olist_geolocation_dataset,1000163,261831,26.18
2,olist_order_items_dataset,112650,0,0.00
3,olist_order_payments_dataset,103886,0,0.00
4,olist_order_reviews_dataset,99224,0,0.00
5,olist_orders_dataset,99441,0,0.00
6,olist_products_dataset,32951,0,0.00
7,olist_sellers_dataset,3095,0,0.00
8,product_category_name_translation(1),71,0,0.00


In [12]:

# Investigate the geolocation dataset

geo = datasets["olist_geolocation_dataset"]

print("Total geolocation rows:", len(geo))

print("Exact duplicate rows:", geo.duplicated().sum())

print("Rows after removing exact duplicates:",
      len(geo.drop_duplicates()))

print("\nUnique postal-code prefixes:",
      geo["geolocation_zip_code_prefix"].nunique())

print("\nTop 10 most frequent postal-code prefixes:")

display(
    geo["geolocation_zip_code_prefix"]
    .value_counts()
    .head(10)
)


Total geolocation rows: 1000163
Exact duplicate rows: 261831
Rows after removing exact duplicates: 738332

Unique postal-code prefixes: 19015

Top 10 most frequent postal-code prefixes:


geolocation_zip_code_prefix
24220    1146
24230    1102
38400     965
35500     907
11680     879
22631     832
30140     810
11740     788
38408     773
28970     743
Name: count, dtype: int64

In [13]:

# PRIMARY KEY UNIQUENESS CHECK

key_checks = {
    "olist_customers_dataset": ["customer_id"],
    "olist_orders_dataset": ["order_id"],
    "olist_products_dataset": ["product_id"],
    "olist_sellers_dataset": ["seller_id"],
    "olist_order_reviews_dataset": ["review_id"],
    "olist_order_items_dataset": [
        "order_id", "order_item_id"
    ],
    "olist_order_payments_dataset": [
        "order_id", "payment_sequential"
    ],
    "product_category_name_translation(1)": [
        "product_category_name"
    ]
}

results = []

for name, keys in key_checks.items():

    df = datasets[name]

    duplicate_keys = df.duplicated(
        subset=keys
    ).sum()

    results.append({
        "Dataset": name,
        "Key Columns": ", ".join(keys),
        "Total Rows": len(df),
        "Duplicate Keys": duplicate_keys,
        "Unique Key": duplicate_keys == 0
    })

key_report = pd.DataFrame(results)

display(key_report)


,Dataset,Key Columns,Total Rows,Duplicate Keys,Unique Key
0,olist_customers_dataset,customer_id,99441,0,True
1,olist_orders_dataset,order_id,99441,0,True
2,olist_products_dataset,product_id,32951,0,True
3,olist_sellers_dataset,seller_id,3095,0,True
4,olist_order_reviews_dataset,review_id,99224,814,False
5,olist_order_items_dataset,"order_id, order_item_id",112650,0,True
6,olist_order_payments_dataset,"order_id, payment_sequential",103886,0,True
7,product_category_name_translation(1),product_category_name,71,0,True


In [14]:

# Investigate repeated review IDs

reviews = datasets["olist_order_reviews_dataset"]

# Find all records with repeated review IDs
repeated_reviews = reviews[
    reviews.duplicated(
        subset=["review_id"],
        keep=False
    )
].copy()

# Count distinct repeated review IDs
print(
    "Number of repeated review IDs:",
    repeated_reviews["review_id"].nunique()
)

# Count all rows involved
print(
    "Total rows involved:",
    len(repeated_reviews)
)

# Check whether repeated IDs link to different orders
review_id_summary = (
    repeated_reviews
    .groupby("review_id")
    .agg(
        record_count=("order_id", "size"),
        unique_orders=("order_id", "nunique"),
        unique_scores=("review_score", "nunique")
    )
)

print("\nRepeated IDs linked to multiple orders:")
print(
    (review_id_summary["unique_orders"] > 1).sum()
)

print("\nRepeated IDs with different review scores:")
print(
    (review_id_summary["unique_scores"] > 1).sum()
)

print("\nSample repeated review records:")
display(
    repeated_reviews.sort_values("review_id").head(10)
)


Number of repeated review IDs: 789
Total rows involved: 1603

Repeated IDs linked to multiple orders:
789

Repeated IDs with different review scores:
0

Sample repeated review records:


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [15]:

# Check uniqueness of review_id + order_id

reviews = datasets["olist_order_reviews_dataset"]

print("Total review records:", len(reviews))

print(
    "Unique review IDs:",
    reviews["review_id"].nunique()
)

print(
    "Unique order IDs:",
    reviews["order_id"].nunique()
)

print(
    "Duplicate review_id + order_id combinations:",
    reviews.duplicated(
        subset=["review_id", "order_id"]
    ).sum()
)

print(
    "Orders with multiple review records:",
    (reviews.groupby("order_id").size() > 1).sum()
)


Total review records: 99224
Unique review IDs: 98410
Unique order IDs: 98673
Duplicate review_id + order_id combinations: 0
Orders with multiple review records: 547


In [16]:

# REFERENTIAL INTEGRITY CHECK

relationship_checks = [
    ("Orders → Customers",
     "olist_orders_dataset", "customer_id",
     "olist_customers_dataset", "customer_id"),

    ("Order Items → Orders",
     "olist_order_items_dataset", "order_id",
     "olist_orders_dataset", "order_id"),

    ("Payments → Orders",
     "olist_order_payments_dataset", "order_id",
     "olist_orders_dataset", "order_id"),

    ("Reviews → Orders",
     "olist_order_reviews_dataset", "order_id",
     "olist_orders_dataset", "order_id"),

    ("Order Items → Products",
     "olist_order_items_dataset", "product_id",
     "olist_products_dataset", "product_id"),

    ("Order Items → Sellers",
     "olist_order_items_dataset", "seller_id",
     "olist_sellers_dataset", "seller_id")
]

integrity_results = []

for label, child_table, child_key, parent_table, parent_key in relationship_checks:

    child = datasets[child_table]
    parent = datasets[parent_table]

    # Check which child IDs are missing from the parent table
    missing_matches = ~child[child_key].isin(parent[parent_key])

    integrity_results.append({
        "Relationship": label,
        "Child Rows": len(child),
        "Unmatched Rows": missing_matches.sum(),
        "Unmatched Unique IDs": child.loc[
            missing_matches, child_key
        ].nunique()
    })

integrity_report = pd.DataFrame(integrity_results)

display(integrity_report)


,Relationship,Child Rows,Unmatched Rows,Unmatched Unique IDs
0,Orders → Customers,99441,0,0
1,Order Items → Orders,112650,0,0
2,Payments → Orders,103886,0,0
3,Reviews → Orders,99224,0,0
4,Order Items → Products,112650,0,0
5,Order Items → Sellers,112650,0,0


In [17]:

# STEP 23: CONVERT DATE COLUMNS

# Create a working copy of our flagged Orders data
orders_clean = orders_clean.copy()

# Identify date columns
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Convert text columns to datetime
for col in date_columns:
    orders_clean[col] = pd.to_datetime(
        orders_clean[col],
        errors="coerce"
    )

# Check the updated data types
print("Updated date column types:\n")

print(orders_clean[date_columns].dtypes)

# Display the first five records
display(orders_clean[date_columns].head())


Updated date column types:

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [18]:

# Compare missing dates before and after conversion

conversion_check = pd.DataFrame({
    "Before Conversion": orders[date_columns].isna().sum(),
    "After Conversion": orders_clean[date_columns].isna().sum()
})

conversion_check["New Missing Values"] = (
    conversion_check["After Conversion"]
    - conversion_check["Before Conversion"]
)

display(conversion_check)


,Before Conversion,After Conversion,New Missing Values
order_purchase_timestamp,0,0,0
order_approved_at,160,160,0
order_delivered_carrier_date,1783,1783,0
order_delivered_customer_date,2965,2965,0
order_estimated_delivery_date,0,0,0


In [19]:

# STEP 25: DELIVERY PERFORMANCE METRICS

# Create a working copy
delivery_analysis = orders_clean.copy()

# Identify orders eligible for delivery analysis
valid_delivery = (
    (delivery_analysis["order_status"] == "delivered") &
    delivery_analysis["order_delivered_customer_date"].notna()
)

# Calculate delivery duration in days
delivery_analysis["delivery_days"] = (
    delivery_analysis["order_delivered_customer_date"]
    - delivery_analysis["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Compare calendar dates because the estimated
# delivery date has no time-of-day information
actual_date = delivery_analysis[
    "order_delivered_customer_date"
].dt.normalize()

estimated_date = delivery_analysis[
    "order_estimated_delivery_date"
].dt.normalize()

# Create an on-time delivery flag
delivery_analysis["on_time_delivery"] = (
    valid_delivery & (actual_date <= estimated_date)
)

# Create a late delivery flag
delivery_analysis["late_delivery"] = (
    valid_delivery & (actual_date > estimated_date)
)

# Calculate our initial metrics
valid_orders = delivery_analysis.loc[valid_delivery]

print("Orders eligible for delivery analysis:", len(valid_orders))

print(
    "Average delivery days:",
    round(valid_orders["delivery_days"].mean(), 2)
)

print(
    "On-time delivery rate:",
    round(valid_orders["on_time_delivery"].mean() * 100, 2),
    "%"
)

print(
    "Late deliveries:",
    int(valid_orders["late_delivery"].sum())
)

print(
    "Negative delivery durations:",
    int((valid_orders["delivery_days"] < 0).sum())
)


Orders eligible for delivery analysis: 96470
Average delivery days: 12.56
On-time delivery rate: 93.23 %
Late deliveries: 6534
Negative delivery durations: 0


In [20]:

# STEP 27: DELIVERY DURATION ANALYSIS

# Work with successfully delivered orders
valid_orders = delivery_analysis.loc[
    (delivery_analysis["order_status"] == "delivered") &
    delivery_analysis["order_delivered_customer_date"].notna()
].copy()

# Calculate delivery statistics
delivery_stats = valid_orders["delivery_days"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

print("DELIVERY DURATION STATISTICS")
display(delivery_stats.round(2))

# Count orders taking longer than 30 days
long_deliveries = (
    valid_orders["delivery_days"] > 30
).sum()

print("Orders taking more than 30 days:", long_deliveries)

print(
    "Percentage taking more than 30 days:",
    round(long_deliveries / len(valid_orders) * 100, 2),
    "%"
)


DELIVERY DURATION STATISTICS


count    96470.00
mean        12.56
std          9.55
min          0.53
25%          6.77
50%         10.22
75%         15.72
90%         23.10
95%         29.27
99%         46.05
max        209.63
Name: delivery_days, dtype: float64

Orders taking more than 30 days: 4550
Percentage taking more than 30 days: 4.72 %


In [21]:

# STEP 29: INVESTIGATE LONG DELIVERY TIMES

# Select the 10 longest deliveries
longest_deliveries = valid_orders.nlargest(
    10, "delivery_days"
)

columns_to_show = [
    "order_id",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_days",
    "late_delivery"
]

display(
    longest_deliveries[columns_to_show]
)

# Compare long deliveries with late deliveries
long_delivery_summary = valid_orders[
    valid_orders["delivery_days"] > 30
]

print(
    "Orders taking over 30 days:",
    len(long_delivery_summary)
)

print(
    "Late among these orders:",
    int(long_delivery_summary["late_delivery"].sum())
)

print(
    "Late percentage among long deliveries:",
    round(
        long_delivery_summary["late_delivery"].mean() * 100,
        2
    ),
    "%"
)


,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,late_delivery
19590,ca07593549f1816d26a572e06dc1eab6,2017-02-21 23:31:27,2017-09-19 14:36:39,2017-03-22,209.628611,True
55619,1b3190b2dfa9d789e1f14c05b647a14a,2018-02-23 14:57:35,2018-09-19 23:24:07,2018-03-15,208.351759,True
61610,440d0d17af552815d15a9e41abe49359,2017-03-07 23:59:51,2017-09-19 15:12:50,2017-04-07,195.634016,True
70307,2fb597c2f772eca01b1f5c561bf6cc7b,2017-03-08 18:09:02,2017-09-19 14:33:17,2017-04-17,194.850174,True
89130,285ab9426d6982034523a855f55a885e,2017-03-08 22:47:40,2017-09-19 14:00:04,2017-04-06,194.633611,True
38509,0f4519c5f1c541ddec9f21b3bddd533a,2017-03-09 13:26:57,2017-09-19 14:38:21,2017-04-11,194.049583,True
11399,47b40429ed8cce3aee9199792275433f,2018-01-03 09:44:01,2018-07-13 20:51:31,2018-01-19,191.463542,True
81401,2fe324febf907e3ea3f2aa9650869fa5,2017-03-13 20:17:10,2017-09-19 17:00:07,2017-04-05,189.863160,True
54480,2d7561026d542c8dbd8f0daeadf67a43,2017-03-15 11:24:27,2017-09-19 14:38:18,2017-04-13,188.134618,True
68769,c27815f7e3dd0b926b58552628481575,2017-03-15 23:23:17,2017-09-19 17:14:25,2017-04-10,187.743843,True


Orders taking over 30 days: 4550
Late among these orders: 3581
Late percentage among long deliveries: 78.7 %


In [22]:

# STEP 31: INVESTIGATE DELIVERY DATE CLUSTERS

long_orders = valid_orders[
    valid_orders["delivery_days"] > 30
].copy()

# Extract calendar delivery date
long_orders["delivery_date"] = (
    long_orders["order_delivered_customer_date"]
    .dt.date
)

# Find the dates with the most long deliveries
print("Top 10 delivery dates for orders taking over 30 days:")

display(
    long_orders["delivery_date"]
    .value_counts()
    .head(10)
)

# Investigate September 19, 2017
target_date = pd.to_datetime("2017-09-19").date()

september_orders = long_orders[
    long_orders["delivery_date"] == target_date
]

print("\nLong deliveries recorded on September 19, 2017:",
      len(september_orders))

print(
    "Average delivery duration:",
    round(september_orders["delivery_days"].mean(), 2)
)

print(
    "Late deliveries:",
    int(september_orders["late_delivery"].sum())
)


Top 10 delivery dates for orders taking over 30 days:


delivery_date
2018-04-03    62
2018-01-04    61
2018-04-10    60
2018-01-08    60
2018-01-09    58
2018-04-16    58
2018-04-11    58
2017-09-19    56
2017-12-27    53
2018-01-03    52
Name: count, dtype: int64


Long deliveries recorded on September 19, 2017: 56
Average delivery duration: 94.54
Late deliveries: 50


In [23]:

# STEP 32: MERGE ORDERS WITH CUSTOMER LOCATIONS

# Select relevant customer information
customers = datasets["olist_customers_dataset"]

customer_locations = customers[
    ["customer_id", "customer_city", "customer_state"]
].copy()

# Merge customer locations into our delivery analysis
regional_delivery = valid_orders.merge(
    customer_locations,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# Check the merge results
print("Orders before merge:", len(valid_orders))
print("Orders after merge:", len(regional_delivery))

print(
    "Missing customer states:",
    regional_delivery["customer_state"].isna().sum()
)

print(
    "Number of customer states:",
    regional_delivery["customer_state"].nunique()
)

display(
    regional_delivery[
        [
            "order_id",
            "customer_city",
            "customer_state",
            "delivery_days",
            "late_delivery"
        ]
    ].head()
)


Orders before merge: 96470
Orders after merge: 96470
Missing customer states: 0
Number of customer states: 27


,order_id,customer_city,customer_state,delivery_days,late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,sao paulo,SP,8.436574,False
1,53cdb2fc8bc7dce0b6741e2150273451,barreiras,BA,13.782037,False
2,47770eb9100c2d0c44946d9cf07ec65d,vianopolis,GO,9.394213,False
3,949d5b44dbf5de918fe9c16f97b45f8a,sao goncalo do amarante,RN,13.208750,False
4,ad21c59c0840e6cb83a9ceb5573f8159,santo andre,SP,2.873877,False


In [24]:

# STEP 33: REGIONAL DELIVERY PERFORMANCE

state_performance = (
    regional_delivery
    .groupby("customer_state")
    .agg(
        total_orders=("order_id", "count"),
        avg_delivery_days=("delivery_days", "mean"),
        median_delivery_days=("delivery_days", "median"),
        late_orders=("late_delivery", "sum"),
        late_delivery_rate=("late_delivery", "mean")
    )
    .reset_index()
)

# Convert late delivery rate into a percentage
state_performance["late_delivery_rate"] *= 100

# Round results for presentation
state_performance = state_performance.round({
    "avg_delivery_days": 2,
    "median_delivery_days": 2,
    "late_delivery_rate": 2
})

# Sort states from slowest to fastest
state_performance = state_performance.sort_values(
    "avg_delivery_days",
    ascending=False
)

print("STATES WITH THE LONGEST AVERAGE DELIVERY TIMES")

display(state_performance.head(10))


STATES WITH THE LONGEST AVERAGE DELIVERY TIMES


,customer_state,total_orders,avg_delivery_days,median_delivery_days,late_orders,late_delivery_rate
21,RR,41,29.39,25.01,5,12.20
3,AP,67,27.19,24.35,2,2.99
2,AM,145,26.43,25.88,4,2.76
1,AL,397,24.54,22.33,85,21.41
13,PA,946,23.77,21.08,106,11.21
9,MA,717,21.57,19.19,125,17.43
24,SE,335,21.52,18.01,51,15.22
5,CE,1279,21.27,18.21,176,13.76
0,AC,80,21.04,18.36,3,3.75
14,PB,517,20.43,18.15,54,10.44


In [25]:

# STEP 35: STATES WITH HIGHEST LATE DELIVERY RATES

# Focus on states with at least 100 delivered orders
reliable_states = state_performance[
    state_performance["total_orders"] >= 100
].copy()

# Rank states by late-delivery percentage
highest_late_states = reliable_states.sort_values(
    "late_delivery_rate",
    ascending=False
)

print("TOP 10 STATES WITH HIGHEST LATE DELIVERY RATES")

display(
    highest_late_states[
        [
            "customer_state",
            "total_orders",
            "avg_delivery_days",
            "late_orders",
            "late_delivery_rate"
        ]
    ].head(10)
)

print(
    "States included in comparison:",
    len(reliable_states)
)


TOP 10 STATES WITH HIGHEST LATE DELIVERY RATES


,customer_state,total_orders,avg_delivery_days,late_orders,late_delivery_rate
1,AL,397,24.54,85,21.41
9,MA,717,21.57,125,17.43
24,SE,335,21.52,51,15.22
16,PI,476,19.46,66,13.87
5,CE,1279,21.27,176,13.76
4,BA,3256,19.34,396,12.16
18,RJ,12350,15.31,1495,12.11
13,PA,946,23.77,106,11.21
7,ES,1995,15.79,214,10.73
14,PB,517,20.43,54,10.44


States included in comparison: 24


In [26]:

# STEP 37: STATES CONTRIBUTING THE MOST LATE ORDERS

# Sort all states by number of late orders
late_volume_states = state_performance.sort_values(
    "late_orders",
    ascending=False
).copy()

# Calculate each state's contribution to all late orders
total_late_orders = state_performance["late_orders"].sum()

late_volume_states["share_of_all_late_orders"] = (
    late_volume_states["late_orders"]
    / total_late_orders * 100
).round(2)

print("TOTAL LATE ORDERS:", total_late_orders)

print("\nTOP 10 STATES BY NUMBER OF LATE DELIVERIES")

display(
    late_volume_states[
        [
            "customer_state",
            "total_orders",
            "late_orders",
            "late_delivery_rate",
            "share_of_all_late_orders"
        ]
    ].head(10)
)


TOTAL LATE ORDERS: 6534

TOP 10 STATES BY NUMBER OF LATE DELIVERIES


,customer_state,total_orders,late_orders,late_delivery_rate,share_of_all_late_orders
25,SP,40494,1820,4.49,27.85
18,RJ,12350,1495,12.11,22.88
10,MG,11354,519,4.57,7.94
4,BA,3256,396,12.16,6.06
22,RS,5344,325,6.08,4.97
23,SC,3546,291,8.21,4.45
7,ES,1995,214,10.73,3.28
17,PR,4923,199,4.04,3.05
5,CE,1279,176,13.76,2.69
15,PE,1593,153,9.60,2.34


In [27]:

# STEP 39: PREPARE CUSTOMER REVIEW DATA

reviews = datasets["olist_order_reviews_dataset"].copy()

# Check review score distribution
print("CUSTOMER REVIEW SCORE DISTRIBUTION")

display(
    reviews["review_score"]
    .value_counts()
    .sort_index()
)

# Check the number of review records per order
reviews_per_order = (
    reviews.groupby("order_id")
    .size()
)

print("\nTotal review records:", len(reviews))

print(
    "Unique orders with reviews:",
    reviews["order_id"].nunique()
)

print(
    "Orders with multiple review records:",
    (reviews_per_order > 1).sum()
)

print(
    "Maximum review records for one order:",
    reviews_per_order.max()
)

# Check for missing review scores
print(
    "Missing review scores:",
    reviews["review_score"].isna().sum()
)


CUSTOMER REVIEW SCORE DISTRIBUTION


review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64


Total review records: 99224
Unique orders with reviews: 98673
Orders with multiple review records: 547
Maximum review records for one order: 3
Missing review scores: 0


In [28]:

# STEP 41: CREATE ORDER-LEVEL REVIEW SCORES

# Aggregate reviews to one row per order
order_reviews = (
    reviews.groupby("order_id")
    .agg(
        avg_review_score=("review_score", "mean"),
        review_count=("review_score", "count")
    )
    .reset_index()
)

# Check the results
print("Original review records:", len(reviews))
print("Order-level review records:", len(order_reviews))

print(
    "Duplicate order IDs after aggregation:",
    order_reviews["order_id"].duplicated().sum()
)

print(
    "Orders with multiple reviews:",
    (order_reviews["review_count"] > 1).sum()
)

display(order_reviews.head(10))


Original review records: 99224
Order-level review records: 98673
Duplicate order IDs after aggregation: 0
Orders with multiple reviews: 547


,order_id,avg_review_score,review_count
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,1
2,000229ec398224ef6ca0657da4fc703e,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1
5,00048cc3ae777c65dbb7d2a0634bc1ea,4.0,1
6,00054e8431b9d7675808bcb819fb4a32,4.0,1
7,000576fe39319847cbb9d288c5617fa6,5.0,1
8,0005a1a1728c9d785b8e2b08b904576c,1.0,1
9,0005f50442cb953dcd1d21e1fb923495,4.0,1


In [29]:

# STEP 42: MERGE REVIEWS WITH DELIVERY PERFORMANCE

# Merge order-level reviews into delivery records
delivery_reviews = regional_delivery.merge(
    order_reviews,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# Validate the merge
print("Orders before merge:", len(regional_delivery))
print("Orders after merge:", len(delivery_reviews))

print(
    "Orders without review scores:",
    delivery_reviews["avg_review_score"].isna().sum()
)

print(
    "Orders with review scores:",
    delivery_reviews["avg_review_score"].notna().sum()
)

print(
    "Duplicate order IDs:",
    delivery_reviews["order_id"].duplicated().sum()
)

# Preview the combined dataset
display(
    delivery_reviews[
        [
            "order_id",
            "customer_state",
            "delivery_days",
            "late_delivery",
            "avg_review_score",
            "review_count"
        ]
    ].head(10)
)


Orders before merge: 96470
Orders after merge: 96470
Orders without review scores: 646
Orders with review scores: 95824
Duplicate order IDs: 0


,order_id,customer_state,delivery_days,late_delivery,avg_review_score,review_count
0,e481f51cbdc54678b7cc49136f2d6af7,SP,8.436574,False,4.0,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,BA,13.782037,False,4.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,GO,9.394213,False,5.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,RN,13.208750,False,5.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,SP,2.873877,False,5.0,1.0
5,a4591c265e18cb1dcee52889e2d8acc3,PR,16.542245,False,4.0,1.0
6,6514b8ad8028c9f2cc2374ded245783f,RJ,9.989826,False,5.0,1.0
7,76c6e866289321a7c93b82b54852dc33,RS,9.818762,False,1.0,1.0
8,e69bfb5eb88e0ed6a785585b27e16dbf,SP,18.221852,False,5.0,1.0
9,e6ce16cb79ec1d90b1da9085a6118aeb,RJ,12.650937,False,1.0,1.0


In [30]:

# STEP 44: DELIVERY PERFORMANCE VS CUSTOMER SATISFACTION

# Keep orders with available review scores
reviewed_deliveries = delivery_reviews[
    delivery_reviews["avg_review_score"].notna()
].copy()

# Create readable delivery status
reviewed_deliveries["delivery_status"] = (
    reviewed_deliveries["late_delivery"]
    .map({
        False: "On Time",
        True: "Late"
    })
)

# Flag low-rated orders
reviewed_deliveries["low_rating"] = (
    reviewed_deliveries["avg_review_score"] <= 2
)

# Compare customer satisfaction
satisfaction_comparison = (
    reviewed_deliveries
    .groupby("delivery_status")
    .agg(
        total_orders=("order_id", "count"),
        avg_review_score=("avg_review_score", "mean"),
        low_rated_orders=("low_rating", "sum"),
        low_rating_rate=("low_rating", "mean")
    )
    .reset_index()
)

# Convert low rating rate into percentage
satisfaction_comparison["low_rating_rate"] *= 100

# Round results
satisfaction_comparison = satisfaction_comparison.round({
    "avg_review_score": 2,
    "low_rating_rate": 2
})

print("CUSTOMER SATISFACTION: ON-TIME VS LATE DELIVERIES")

display(satisfaction_comparison)

print(
    "Total reviewed deliveries:",
    satisfaction_comparison["total_orders"].sum()
)


CUSTOMER SATISFACTION: ON-TIME VS LATE DELIVERIES


,delivery_status,total_orders,avg_review_score,low_rated_orders,low_rating_rate
0,Late,6381,2.27,3979,62.36
1,On Time,89443,4.29,8257,9.23


Total reviewed deliveries: 95824


In [31]:

# STEP 46: CALCULATE DAYS LATE

# Create a working copy
delay_analysis = reviewed_deliveries.copy()

# Calculate calendar-day difference between
# actual and estimated delivery dates
delay_analysis["days_late"] = (
    delay_analysis["order_delivered_customer_date"].dt.normalize()
    - delay_analysis["order_estimated_delivery_date"].dt.normalize()
).dt.days

# Keep only late deliveries
late_reviewed_orders = delay_analysis[
    delay_analysis["days_late"] > 0
].copy()

# Examine delay duration
print("LATE DELIVERY DURATION STATISTICS")

display(
    late_reviewed_orders["days_late"].describe().round(2)
)

# Check consistency with previous analysis
print(
    "Late reviewed orders:",
    len(late_reviewed_orders)
)

print(
    "Average review score for late orders:",
    round(late_reviewed_orders["avg_review_score"].mean(), 2)
)


LATE DELIVERY DURATION STATISTICS


count    6381.00
mean       10.52
std        14.46
min         1.00
25%         3.00
50%         7.00
75%        13.00
max       188.00
Name: days_late, dtype: float64

Late reviewed orders: 6381
Average review score for late orders: 2.27


In [32]:

# STEP 48: CUSTOMER SATISFACTION BY DELAY DURATION

# Create delay categories
late_reviewed_orders["delay_category"] = pd.cut(
    late_reviewed_orders["days_late"],
    bins=[0, 3, 7, 14, float("inf")],
    labels=[
        "1-3 days",
        "4-7 days",
        "8-14 days",
        "15+ days"
    ]
)

# Calculate customer satisfaction for each category
delay_satisfaction = (
    late_reviewed_orders
    .groupby("delay_category", observed=True)
    .agg(
        total_orders=("order_id", "count"),
        avg_review_score=("avg_review_score", "mean"),
        low_rated_orders=("low_rating", "sum"),
        low_rating_rate=("low_rating", "mean")
    )
    .reset_index()
)

# Convert low-rating rate into percentage
delay_satisfaction["low_rating_rate"] *= 100

# Round results
delay_satisfaction = delay_satisfaction.round({
    "avg_review_score": 2,
    "low_rating_rate": 2
})

print("CUSTOMER SATISFACTION BY DELIVERY DELAY")

display(delay_satisfaction)

# Validate that all late reviewed orders are included
print(
    "Total orders analysed:",
    delay_satisfaction["total_orders"].sum()
)


CUSTOMER SATISFACTION BY DELIVERY DELAY


,delay_category,total_orders,avg_review_score,low_rated_orders,low_rating_rate
0,1-3 days,1852,3.29,596,32.18
1,4-7 days,1748,2.11,1181,67.56
2,8-14 days,1446,1.67,1158,80.08
3,15+ days,1335,1.73,1044,78.20


Total orders analysed: 6381


In [33]:

# STEP 50: INSPECT ORDER ITEMS DATASET

order_items = datasets["olist_order_items_dataset"].copy()

print("ORDER ITEMS DATASET")

print("Rows and columns:", order_items.shape)

print("\nColumn names:")
print(order_items.columns.tolist())

print("\nData types:")
display(order_items.dtypes)

print("\nMissing values:")
display(order_items.isna().sum())

print("\nPrice and freight statistics:")
display(
    order_items[["price", "freight_value"]]
    .describe()
    .round(2)
)

print("\nFirst five records:")
display(order_items.head())


ORDER ITEMS DATASET
Rows and columns: (112650, 7)

Column names:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Data types:


order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object


Missing values:


order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64


Price and freight statistics:


,price,freight_value
count,112650.00,112650.00
mean,120.65,19.99
std,183.63,15.81
min,0.85,0.00
25%,39.90,13.08
50%,74.99,16.26
75%,134.90,21.15
max,6735.00,409.68



First five records:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [34]:

# STEP 52: SALES AND FREIGHT ANALYSIS

# Create a working copy
sales_items = order_items.copy()

# Calculate total merchandise sales value
total_sales = sales_items["price"].sum()

# Calculate total freight charges
total_freight = sales_items["freight_value"].sum()

# Calculate combined value
total_combined = total_sales + total_freight

# Count unique orders
unique_orders = sales_items["order_id"].nunique()

# Calculate average merchandise value per order
avg_order_value = total_sales / unique_orders

# Display results
print("E-COMMERCE SALES SUMMARY")

print(f"Total merchandise sales: R$ {total_sales:,.2f}")

print(f"Total freight charges: R$ {total_freight:,.2f}")

print(f"Combined merchandise + freight: R$ {total_combined:,.2f}")

print(f"Unique orders: {unique_orders:,}")

print(f"Average merchandise value per order: R$ {avg_order_value:,.2f}")

# Check item counts per order
items_per_order = sales_items.groupby("order_id").size()

print("\nMaximum item records in one order:", items_per_order.max())

print(
    "Orders with multiple item records:",
    (items_per_order > 1).sum()
)


E-COMMERCE SALES SUMMARY
Total merchandise sales: R$ 13,591,643.70
Total freight charges: R$ 2,251,909.54
Combined merchandise + freight: R$ 15,843,553.24
Unique orders: 98,666
Average merchandise value per order: R$ 137.75

Maximum item records in one order: 21
Orders with multiple item records: 9803


In [35]:

# STEP 54: AGGREGATE SALES TO ORDER LEVEL

order_sales = (
    sales_items
    .groupby("order_id")
    .agg(
        total_items=("order_item_id", "count"),
        merchandise_value=("price", "sum"),
        freight_value=("freight_value", "sum")
    )
    .reset_index()
)

# Calculate combined order value
order_sales["combined_order_value"] = (
    order_sales["merchandise_value"]
    + order_sales["freight_value"]
)

# Validate the aggregation
print("ORDER-LEVEL SALES VALIDATION")

print("Original item records:", len(sales_items))

print("Aggregated orders:", len(order_sales))

print(
    "Duplicate order IDs:",
    order_sales["order_id"].duplicated().sum()
)

print(
    "Merchandise total:",
    round(order_sales["merchandise_value"].sum(), 2)
)

print(
    "Freight total:",
    round(order_sales["freight_value"].sum(), 2)
)

print("\nFIRST FIVE AGGREGATED ORDERS")

display(order_sales.head())


ORDER-LEVEL SALES VALIDATION
Original item records: 112650
Aggregated orders: 98666
Duplicate order IDs: 0
Merchandise total: 13591643.7
Freight total: 2251909.54

FIRST FIVE AGGREGATED ORDERS


,order_id,total_items,merchandise_value,freight_value,combined_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,218.04


In [36]:

# STEP 56: MERGE ORDER SALES WITH ORDER INFORMATION

# Merge order-level sales into cleaned orders
sales_analysis = orders_clean.merge(
    order_sales,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

# Validate the merge
print("SALES AND ORDERS MERGE VALIDATION")

print("Orders before merge:", len(orders_clean))

print("Orders after merge:", len(sales_analysis))

print(
    "Orders without item records:",
    sales_analysis["merchandise_value"].isna().sum()
)

print(
    "Duplicate order IDs:",
    sales_analysis["order_id"].duplicated().sum()
)

print("\nMerge results:")
display(sales_analysis["_merge"].value_counts())

print("\nOrder status distribution:")
display(sales_analysis["order_status"].value_counts())

print("\nFirst five merged records:")
display(
    sales_analysis[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "total_items",
            "merchandise_value",
            "freight_value",
            "combined_order_value"
        ]
    ].head()
)


SALES AND ORDERS MERGE VALIDATION
Orders before merge: 99441
Orders after merge: 99441
Orders without item records: 775
Duplicate order IDs: 0

Merge results:


_merge
both          98666
left_only       775
right_only        0
Name: count, dtype: int64


Order status distribution:


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


First five merged records:


,order_id,order_status,order_purchase_timestamp,total_items,merchandise_value,freight_value,combined_order_value
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-02 10:56:33,1.0,29.99,8.72,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24 20:41:37,1.0,118.70,22.76,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-08 08:38:49,1.0,159.90,19.22,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11-18 19:28:06,1.0,45.00,27.20,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-13 21:18:39,1.0,19.90,8.72,28.62


In [37]:

# STEP 58: SALES PERFORMANCE BY ORDER STATUS

# Group sales by order status
sales_by_status = (
    sales_analysis
    .groupby("order_status")
    .agg(
        total_orders=("order_id", "count"),
        orders_with_sales=("merchandise_value", "count"),
        merchandise_value=("merchandise_value", "sum"),
        freight_value=("freight_value", "sum")
    )
    .reset_index()
)

# Calculate combined value
sales_by_status["combined_value"] = (
    sales_by_status["merchandise_value"]
    + sales_by_status["freight_value"]
)

# Calculate average merchandise value per order
# Only include orders with available item records
sales_by_status["avg_order_value"] = (
    sales_by_status["merchandise_value"]
    / sales_by_status["orders_with_sales"]
)

# Sort by merchandise value
sales_by_status = sales_by_status.sort_values(
    "merchandise_value",
    ascending=False
)

# Round financial metrics
sales_by_status = sales_by_status.round(2)

print("MERCHANDISE SALES BY ORDER STATUS")

display(sales_by_status)

# Validate the merchandise total
print(
    "\nTotal merchandise value:",
    round(sales_by_status["merchandise_value"].sum(), 2)
)

print(
    "Orders with sales records:",
    sales_by_status["orders_with_sales"].sum()
)


MERCHANDISE SALES BY ORDER STATUS


,order_status,total_orders,orders_with_sales,merchandise_value,freight_value,combined_value,avg_order_value
3,delivered,96478,96478,13221498.11,2198275.64,15419773.75,137.04
6,shipped,1107,1106,150727.44,26401.90,177129.34,136.28
1,canceled,625,461,95235.27,10650.45,105885.72,206.58
4,invoiced,314,312,61526.37,7462.38,68988.75,197.20
5,processing,301,301,60439.22,8954.89,69394.11,200.79
7,unavailable,609,6,2007.69,132.80,2140.49,334.62
0,approved,2,2,209.60,31.48,241.08,104.80
2,created,5,0,0.00,0.00,0.00,NaN



Total merchandise value: 13591643.7
Orders with sales records: 98666


In [38]:

# STEP 60: MONTHLY DELIVERED-ORDER SALES ANALYSIS

# Keep delivered orders only
delivered_sales = sales_analysis[
    sales_analysis["order_status"] == "delivered"
].copy()

# Create purchase month
delivered_sales["purchase_month"] = (
    delivered_sales["order_purchase_timestamp"]
    .dt.to_period("M")
)

# Aggregate monthly performance
monthly_sales = (
    delivered_sales
    .groupby("purchase_month")
    .agg(
        total_orders=("order_id", "count"),
        merchandise_value=("merchandise_value", "sum"),
        freight_value=("freight_value", "sum")
    )
    .reset_index()
)

# Calculate average order value
monthly_sales["avg_order_value"] = (
    monthly_sales["merchandise_value"]
    / monthly_sales["total_orders"]
)

# Calculate month-over-month sales growth
monthly_sales["mom_growth_pct"] = (
    monthly_sales["merchandise_value"]
    .pct_change(fill_method=None) * 100
)

# Format purchase month for display
monthly_sales["purchase_month"] = (
    monthly_sales["purchase_month"].astype(str)
)

# Round financial values
monthly_sales = monthly_sales.round({
    "merchandise_value": 2,
    "freight_value": 2,
    "avg_order_value": 2,
    "mom_growth_pct": 2
})

print("MONTHLY DELIVERED-ORDER SALES PERFORMANCE")

display(monthly_sales)

# Validate totals
print("\nVALIDATION")

print("Total delivered orders:", monthly_sales["total_orders"].sum())

print(
    "Total delivered merchandise value:",
    round(monthly_sales["merchandise_value"].sum(), 2)
)

print("Number of months:", len(monthly_sales))


MONTHLY DELIVERED-ORDER SALES PERFORMANCE


,purchase_month,total_orders,merchandise_value,freight_value,avg_order_value,mom_growth_pct
0,2016-09,1,134.97,8.49,134.97,NaN
1,2016-10,265,40325.11,6165.55,152.17,29777.09
2,2016-12,1,10.90,8.72,10.90,-99.97
3,2017-01,750,111798.36,15684.01,149.06,1025573.03
4,2017-02,1653,234223.40,37015.92,141.70,109.51
5,2017-03,2546,359198.85,55132.10,141.08,53.36
6,2017-04,2303,340669.68,50142.72,147.92,-5.16
7,2017-05,3546,489338.25,77513.15,138.00,43.64
8,2017-06,3135,421923.37,68127.00,134.58,-13.78
9,2017-07,3872,481604.52,84694.56,124.38,14.15



VALIDATION
Total delivered orders: 96478
Total delivered merchandise value: 13221498.11
Number of months: 23


In [39]:

# STEP 62: CHECK SALES REPORTING PERIOD

# Check earliest and latest purchase dates
print("DELIVERED SALES DATE COVERAGE")

print(
    "Earliest purchase date:",
    delivered_sales["order_purchase_timestamp"].min()
)

print(
    "Latest purchase date:",
    delivered_sales["order_purchase_timestamp"].max()
)

# Generate every calendar month between
# the earliest and latest purchase dates
all_months = pd.period_range(
    start=delivered_sales["order_purchase_timestamp"].min(),
    end=delivered_sales["order_purchase_timestamp"].max(),
    freq="M"
)

# Identify months without delivered-order records
recorded_months = pd.PeriodIndex(
    monthly_sales["purchase_month"],
    freq="M"
)

missing_months = all_months.difference(recorded_months)

print("\nMonths with delivered orders:", len(recorded_months))

print("Missing calendar months:", missing_months.astype(str).tolist())

# Inspect daily order coverage at both ends
daily_orders = (
    delivered_sales
    .groupby(
        delivered_sales["order_purchase_timestamp"].dt.date
    )
    .size()
)

print("\nFIRST 10 DAYS WITH DELIVERED ORDERS")
display(daily_orders.head(10))

print("\nLAST 10 DAYS WITH DELIVERED ORDERS")
display(daily_orders.tail(10))


DELIVERED SALES DATE COVERAGE
Earliest purchase date: 2016-09-15 12:16:38
Latest purchase date: 2018-08-29 15:00:37

Months with delivered orders: 23
Missing calendar months: ['2016-11']

FIRST 10 DAYS WITH DELIVERED ORDERS


order_purchase_timestamp
2016-09-15     1
2016-10-03     7
2016-10-04    54
2016-10-05    35
2016-10-06    41
2016-10-07    38
2016-10-08    36
2016-10-09    20
2016-10-10    34
2016-12-23     1
dtype: int64


LAST 10 DAYS WITH DELIVERED ORDERS


order_purchase_timestamp
2018-08-20    251
2018-08-21    238
2018-08-22    185
2018-08-23    142
2018-08-24     96
2018-08-25     69
2018-08-26     73
2018-08-27     66
2018-08-28     39
2018-08-29     11
dtype: int64

In [40]:

# STEP 64: YEAR-OVER-YEAR SALES COMPARISON

# Create a copy of monthly sales
yoy_sales = monthly_sales.copy()

# Convert month strings to datetime
yoy_sales["purchase_month"] = pd.to_datetime(
    yoy_sales["purchase_month"]
)

# Extract year and month
yoy_sales["year"] = yoy_sales["purchase_month"].dt.year
yoy_sales["month"] = yoy_sales["purchase_month"].dt.month

# Keep matching months: January to July
yoy_sales = yoy_sales[
    (yoy_sales["year"].isin([2017, 2018])) &
    (yoy_sales["month"].between(1, 7))
].copy()

# Compare total merchandise value by year
year_comparison = (
    yoy_sales.groupby("year")
    .agg(
        total_orders=("total_orders", "sum"),
        merchandise_value=("merchandise_value", "sum"),
        freight_value=("freight_value", "sum")
    )
    .reset_index()
)

# Calculate average merchandise value per order
year_comparison["avg_order_value"] = (
    year_comparison["merchandise_value"] /
    year_comparison["total_orders"]
)

# Calculate year-over-year growth
sales_2017 = year_comparison.loc[
    year_comparison["year"] == 2017,
    "merchandise_value"
].iloc[0]

sales_2018 = year_comparison.loc[
    year_comparison["year"] == 2018,
    "merchandise_value"
].iloc[0]

yoy_growth = (
    (sales_2018 - sales_2017) / sales_2017
) * 100

print("YEAR-OVER-YEAR SALES COMPARISON")
print("Period: January-July")

display(year_comparison.round(2))

print(f"\nYear-over-year merchandise growth: {yoy_growth:.2f}%")


YEAR-OVER-YEAR SALES COMPARISON
Period: January-July


,year,total_orders,merchandise_value,freight_value,avg_order_value
0,2017,17805,2438756.43,388309.46,136.97
1,2018,46432,6379548.48,1086544.65,137.40



Year-over-year merchandise growth: 161.59%


In [41]:

# STEP 66: MONTHLY YEAR-OVER-YEAR SALES GROWTH

# Create a monthly comparison table
monthly_yoy = yoy_sales.pivot(
    index="month",
    columns="year",
    values="merchandise_value"
).reset_index()

# Rename columns
monthly_yoy.columns = [
    "month",
    "sales_2017",
    "sales_2018"
]

# Calculate absolute sales increase
monthly_yoy["sales_increase"] = (
    monthly_yoy["sales_2018"]
    - monthly_yoy["sales_2017"]
)

# Calculate percentage growth
monthly_yoy["yoy_growth_pct"] = (
    monthly_yoy["sales_increase"]
    / monthly_yoy["sales_2017"]
) * 100

# Convert month numbers to names
monthly_yoy["month_name"] = pd.to_datetime(
    monthly_yoy["month"],
    format="%m"
).dt.strftime("%B")

# Arrange columns
monthly_yoy = monthly_yoy[
    [
        "month_name",
        "sales_2017",
        "sales_2018",
        "sales_increase",
        "yoy_growth_pct"
    ]
]

print("MONTHLY YEAR-OVER-YEAR SALES COMPARISON")

display(monthly_yoy.round(2))

# Identify month with largest absolute increase
largest_increase = monthly_yoy.loc[
    monthly_yoy["sales_increase"].idxmax()
]

print("\nMONTH WITH LARGEST SALES INCREASE")
print("Month:", largest_increase["month_name"])
print(
    "Increase: R$",
    f'{largest_increase["sales_increase"]:,.2f}'
)


MONTHLY YEAR-OVER-YEAR SALES COMPARISON


,month_name,sales_2017,sales_2018,sales_increase,yoy_growth_pct
0,January,111798.36,924645.00,812846.64,727.06
1,February,234223.40,826437.13,592213.73,252.84
2,March,359198.85,953356.25,594157.40,165.41
3,April,340669.68,973534.09,632864.41,185.77
4,May,489338.25,977544.69,488206.44,99.77
5,June,421923.37,856077.86,434154.49,102.90
6,July,481604.52,867953.46,386348.94,80.22



MONTH WITH LARGEST SALES INCREASE
Month: January
Increase: R$ 812,846.64


In [42]:

# STEP 68: INSPECT PRODUCT AND CATEGORY DATA

# Load products dataset
products = datasets["olist_products_dataset"].copy()

# Load category translation dataset
category_translation = datasets[
    "product_category_name_translation(1)"
].copy()

print("PRODUCTS DATASET")
print("Shape:", products.shape)

print("\nProduct columns:")
print(products.columns.tolist())

print("\nMissing values:")
display(products.isna().sum())

print("\nDuplicate product IDs:")
print(products["product_id"].duplicated().sum())

print("\nFirst five products:")
display(products.head())

print("\nCATEGORY TRANSLATION DATASET")
print("Shape:", category_translation.shape)

print("\nTranslation columns:")
print(category_translation.columns.tolist())

print("\nMissing values:")
display(category_translation.isna().sum())

print("\nDuplicate category names:")
print(
    category_translation[
        "product_category_name"
    ].duplicated().sum()
)

print("\nFirst five translations:")
display(category_translation.head())


PRODUCTS DATASET
Shape: (32951, 9)

Product columns:
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

Missing values:


product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


Duplicate product IDs:
0

First five products:


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0



CATEGORY TRANSLATION DATASET
Shape: (71, 2)

Translation columns:
['product_category_name', 'product_category_name_english']

Missing values:


product_category_name            0
product_category_name_english    0
dtype: int64


Duplicate category names:
0

First five translations:


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [43]:

# STEP 70: MERGE PRODUCTS WITH CATEGORY TRANSLATIONS

# Merge products with English category names
products_clean = products.merge(
    category_translation,
    on="product_category_name",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Validate the merge
print("PRODUCT CATEGORY MERGE VALIDATION")

print("Products before merge:", len(products))

print("Products after merge:", len(products_clean))

print(
    "Duplicate product IDs:",
    products_clean["product_id"].duplicated().sum()
)

print("\nMerge results:")
display(products_clean["_merge"].value_counts())

# Check categories without English translations
untranslated = products_clean[
    products_clean["product_category_name"].notna()
    & products_clean["product_category_name_english"].isna()
]

print("\nProducts with untranslated categories:", len(untranslated))

print("\nUntranslated category names:")
display(
    untranslated["product_category_name"]
    .value_counts()
)

# Preserve missing and untranslated categories
products_clean["category_english"] = (
    products_clean["product_category_name_english"]
    .fillna("Unknown / Untranslated")
)

print("\nProducts with unknown/untranslated categories:")
print(
    (products_clean["category_english"] == "Unknown / Untranslated").sum()
)

# Display sample records
display(
    products_clean[
        [
            "product_id",
            "product_category_name",
            "category_english"
        ]
    ].head(10)
)


PRODUCT CATEGORY MERGE VALIDATION
Products before merge: 32951
Products after merge: 32951
Duplicate product IDs: 0

Merge results:


_merge
both          32328
left_only       623
right_only        0
Name: count, dtype: int64


Products with untranslated categories: 13

Untranslated category names:


product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: int64


Products with unknown/untranslated categories:
623


,product_id,product_category_name,category_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares
5,41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,musical_instruments
6,732bd381ad09e530fe0a5f457d81becb,cool_stuff,cool_stuff
7,2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,furniture_decor
8,37cc742be07708b53a98702e77a21a02,eletrodomesticos,home_appliances
9,8c92109888e8cdf9d66dc7e463025574,brinquedos,toys


In [44]:

# STEP 72: MERGE DELIVERED ITEMS WITH PRODUCT CATEGORIES

# Get IDs of delivered orders
delivered_order_ids = delivered_sales["order_id"]

# Keep items belonging to delivered orders
delivered_items = sales_items[
    sales_items["order_id"].isin(delivered_order_ids)
].copy()

# Select product information needed for category analysis
product_categories = products_clean[
    ["product_id", "category_english"]
].copy()

# Merge item-level sales with product categories
category_sales_items = delivered_items.merge(
    product_categories,
    on="product_id",
    how="left",
    validate="many_to_one"
)

# Label any unmatched product IDs
category_sales_items["category_english"] = (
    category_sales_items["category_english"]
    .fillna("Unknown / Untranslated")
)

print("DELIVERED ITEM CATEGORY MERGE VALIDATION")

print("Delivered items before merge:", len(delivered_items))

print("Delivered items after merge:", len(category_sales_items))

print(
    "Duplicate order-item keys:",
    category_sales_items[
        ["order_id", "order_item_id"]
    ].duplicated().sum()
)

print(
    "Items with unknown/untranslated categories:",
    (
        category_sales_items["category_english"]
        == "Unknown / Untranslated"
    ).sum()
)

print("\nFINANCIAL VALIDATION")

print(
    "Delivered merchandise value:",
    round(category_sales_items["price"].sum(), 2)
)

print(
    "Delivered freight charges:",
    round(category_sales_items["freight_value"].sum(), 2)
)

print("\nFIRST FIVE RECORDS")

display(
    category_sales_items[
        [
            "order_id",
            "order_item_id",
            "product_id",
            "category_english",
            "price",
            "freight_value"
        ]
    ].head()
)


DELIVERED ITEM CATEGORY MERGE VALIDATION
Delivered items before merge: 110197
Delivered items after merge: 110197
Duplicate order-item keys: 0
Items with unknown/untranslated categories: 1559

FINANCIAL VALIDATION
Delivered merchandise value: 13221498.11
Delivered freight charges: 2198275.64

FIRST FIVE RECORDS


,order_id,order_item_id,product_id,category_english,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,pet_shop,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,furniture_decor,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,perfumery,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,garden_tools,199.90,18.14


In [45]:

# STEP 74: PRODUCT CATEGORY SALES PERFORMANCE

# Aggregate merchandise value by category
category_performance = (
    category_sales_items
    .groupby("category_english")
    .agg(
        total_items=("order_item_id", "count"),
        total_orders=("order_id", "nunique"),
        merchandise_value=("price", "sum"),
        avg_item_price=("price", "mean"),
        freight_value=("freight_value", "sum")
    )
    .reset_index()
)

# Calculate percentage contribution to total merchandise value
category_performance["sales_share_pct"] = (
    category_performance["merchandise_value"]
    / category_performance["merchandise_value"].sum()
) * 100

# Sort categories by merchandise value
category_performance = category_performance.sort_values(
    "merchandise_value",
    ascending=False
)

# Round results for display
category_performance = category_performance.round({
    "merchandise_value": 2,
    "avg_item_price": 2,
    "freight_value": 2,
    "sales_share_pct": 2
})

print("TOP 10 PRODUCT CATEGORIES BY MERCHANDISE VALUE")

display(category_performance.head(10))

print("\nCATEGORY ANALYSIS VALIDATION")

print(
    "Total merchandise value:",
    round(category_performance["merchandise_value"].sum(), 2)
)

print(
    "Total item records:",
    category_performance["total_items"].sum()
)

print(
    "Number of category groups:",
    len(category_performance)
)

print("\nUNKNOWN CATEGORY PERFORMANCE")

display(
    category_performance[
        category_performance["category_english"]
        == "Unknown / Untranslated"
    ]
)


TOP 10 PRODUCT CATEGORIES BY MERCHANDISE VALUE


,category_english,total_items,total_orders,merchandise_value,avg_item_price,freight_value,sales_share_pct
44,health_beauty,9465,8647,1233131.72,130.28,178957.81,9.33
71,watches_gifts,5859,5495,1166176.98,199.04,98156.14,8.82
8,bed_bath_table,10953,9272,1023434.76,93.44,201774.50,7.74
66,sports_leisure,8431,7530,954852.55,113.25,163404.36,7.22
16,computers_accessories,7644,6530,888724.61,116.26,143999.16,6.72
40,furniture_decor,8160,6307,711927.69,87.25,168402.23,5.38
50,housewares,6795,5743,615628.69,90.60,142763.56,4.66
21,cool_stuff,3718,3559,610204.10,164.12,81476.79,4.62
6,auto,4140,3810,578966.65,139.85,90488.10,4.38
70,toys,4030,3804,471286.48,116.94,75774.58,3.56



CATEGORY ANALYSIS VALIDATION
Total merchandise value: 13221498.11
Total item records: 110197
Number of category groups: 72

UNKNOWN CATEGORY PERFORMANCE


,category_english,total_items,total_orders,merchandise_value,avg_item_price,freight_value,sales_share_pct
0,Unknown / Untranslated,1559,1412,175967.21,112.87,27386.63,1.33


In [46]:

# STEP 75: DISPLAY TOP 10 PRODUCT CATEGORIES

print("TOP 10 PRODUCT CATEGORIES BY MERCHANDISE VALUE")

display(
    category_performance[
        [
            "category_english",
            "total_items",
            "total_orders",
            "merchandise_value",
            "avg_item_price",
            "sales_share_pct"
        ]
    ].head(10)
)

# Calculate combined contribution of top 10 categories
top_10_share = (
    category_performance
    .head(10)["merchandise_value"]
    .sum()
    / category_performance["merchandise_value"].sum()
) * 100

print(
    f"\nTop 10 categories contribute "
    f"{top_10_share:.2f}% of delivered merchandise value."
)


TOP 10 PRODUCT CATEGORIES BY MERCHANDISE VALUE


,category_english,total_items,total_orders,merchandise_value,avg_item_price,sales_share_pct
44,health_beauty,9465,8647,1233131.72,130.28,9.33
71,watches_gifts,5859,5495,1166176.98,199.04,8.82
8,bed_bath_table,10953,9272,1023434.76,93.44,7.74
66,sports_leisure,8431,7530,954852.55,113.25,7.22
16,computers_accessories,7644,6530,888724.61,116.26,6.72
40,furniture_decor,8160,6307,711927.69,87.25,5.38
50,housewares,6795,5743,615628.69,90.60,4.66
21,cool_stuff,3718,3559,610204.10,164.12,4.62
6,auto,4140,3810,578966.65,139.85,4.38
70,toys,4030,3804,471286.48,116.94,3.56



Top 10 categories contribute 62.43% of delivered merchandise value.


In [47]:

# STEP 76: TOP PRODUCT CATEGORIES BY ITEM VOLUME

# Sort categories by number of delivered item records
top_volume_categories = (
    category_performance
    .sort_values(
        "total_items",
        ascending=False
    )
    .head(10)
    .copy()
)

print("TOP 10 CATEGORIES BY DELIVERED ITEM VOLUME")

display(
    top_volume_categories[
        [
            "category_english",
            "total_items",
            "total_orders",
            "merchandise_value",
            "avg_item_price",
            "sales_share_pct"
        ]
    ]
)

# Calculate top 10 contribution to item volume
volume_share = (
    top_volume_categories["total_items"].sum()
    / category_performance["total_items"].sum()
) * 100

print(
    f"\nTop 10 categories account for "
    f"{volume_share:.2f}% of delivered item records."
)

# Compare leaders
highest_value_category = category_performance.iloc[0]

highest_volume_category = top_volume_categories.iloc[0]

print("\nCATEGORY LEADERS")

print(
    "Highest merchandise value:",
    highest_value_category["category_english"]
)

print(
    "Highest item volume:",
    highest_volume_category["category_english"]
)


TOP 10 CATEGORIES BY DELIVERED ITEM VOLUME


,category_english,total_items,total_orders,merchandise_value,avg_item_price,sales_share_pct
8,bed_bath_table,10953,9272,1023434.76,93.44,7.74
44,health_beauty,9465,8647,1233131.72,130.28,9.33
66,sports_leisure,8431,7530,954852.55,113.25,7.22
40,furniture_decor,8160,6307,711927.69,87.25,5.38
16,computers_accessories,7644,6530,888724.61,116.26,6.72
50,housewares,6795,5743,615628.69,90.60,4.66
71,watches_gifts,5859,5495,1166176.98,199.04,8.82
69,telephony,4430,4093,309860.23,69.95,2.34
43,garden_tools,4268,3448,470495.28,110.24,3.56
6,auto,4140,3810,578966.65,139.85,4.38



Top 10 categories account for 63.65% of delivered item records.

CATEGORY LEADERS
Highest merchandise value: health_beauty
Highest item volume: bed_bath_table


In [48]:

# STEP 77: CATEGORIES WITH HIGHEST AVERAGE ITEM PRICES

# Filter categories with at least 100 delivered item records
eligible_categories = category_performance[
    category_performance["total_items"] >= 100
].copy()

# Rank by average item price
highest_price_categories = (
    eligible_categories
    .sort_values("avg_item_price", ascending=False)
    .head(10)
)

print("TOP 10 CATEGORIES BY AVERAGE ITEM PRICE")
print("Minimum requirement: 100 delivered item records")

display(
    highest_price_categories[
        [
            "category_english",
            "total_items",
            "total_orders",
            "avg_item_price",
            "merchandise_value",
            "sales_share_pct"
        ]
    ]
)

print("\nCATEGORY COVERAGE")

print(
    "Total category groups:",
    len(category_performance)
)

print(
    "Categories with at least 100 items:",
    len(eligible_categories)
)

print(
    "Categories with fewer than 100 items:",
    len(category_performance) - len(eligible_categories)
)

print("\nOVERALL AVERAGE DELIVERED ITEM PRICE")

overall_avg_price = category_sales_items["price"].mean()

print(f"R${overall_avg_price:,.2f}")


TOP 10 CATEGORIES BY AVERAGE ITEM PRICE
Minimum requirement: 100 delivered item records


,category_english,total_items,total_orders,avg_item_price,merchandise_value,sales_share_pct
15,computers,199,177,1098.92,218684.14,1.65
46,home_appliances_2,231,227,467.33,107953.95,0.82
1,agro_industry_and_commerce,206,177,342.55,70566.10,0.53
57,musical_instruments,651,611,283.13,184315.74,1.39
64,small_appliances,658,609,277.74,182754.12,1.38
35,fixed_telephony,255,212,216.92,55315.21,0.42
20,construction_tools_safety,183,159,211.88,38773.22,0.29
71,watches_gifts,5859,5495,199.04,1166176.98,8.82
39,furniture_bedroom,103,90,184.97,19051.80,0.14
2,air_conditioning,289,246,184.51,53323.56,0.40



CATEGORY COVERAGE
Total category groups: 72
Categories with at least 100 items: 54
Categories with fewer than 100 items: 18

OVERALL AVERAGE DELIVERED ITEM PRICE
R$119.98


In [49]:

# STEP 78: DISPLAY HIGHEST-PRICED PRODUCT CATEGORIES

print("TOP 10 CATEGORIES BY AVERAGE ITEM PRICE")

display(
    highest_price_categories[
        [
            "category_english",
            "total_items",
            "total_orders",
            "avg_item_price",
            "merchandise_value",
            "sales_share_pct"
        ]
    ].reset_index(drop=True)
)

print("\nOVERALL AVERAGE ITEM PRICE")
print(f"R${overall_avg_price:,.2f}")


TOP 10 CATEGORIES BY AVERAGE ITEM PRICE


,category_english,total_items,total_orders,avg_item_price,merchandise_value,sales_share_pct
0,computers,199,177,1098.92,218684.14,1.65
1,home_appliances_2,231,227,467.33,107953.95,0.82
2,agro_industry_and_commerce,206,177,342.55,70566.10,0.53
3,musical_instruments,651,611,283.13,184315.74,1.39
4,small_appliances,658,609,277.74,182754.12,1.38
5,fixed_telephony,255,212,216.92,55315.21,0.42
6,construction_tools_safety,183,159,211.88,38773.22,0.29
7,watches_gifts,5859,5495,199.04,1166176.98,8.82
8,furniture_bedroom,103,90,184.97,19051.80,0.14
9,air_conditioning,289,246,184.51,53323.56,0.40



OVERALL AVERAGE ITEM PRICE
R$119.98


In [50]:

from pathlib import Path

# Create cleaned data folder
output_folder = Path("../Data/cleaned")
output_folder.mkdir(parents=True, exist_ok=True)

# Export cleaned datasets
orders_clean.to_csv(
    output_folder / "orders_clean.csv",
    index=False
)

products_clean.to_csv(
    output_folder / "products_clean.csv",
    index=False
)

category_sales_items.to_csv(
    output_folder / "category_sales_items.csv",
    index=False
)

delivery_reviews.to_csv(
    output_folder / "delivery_reviews.csv",
    index=False
)

print("CLEANED DATASETS EXPORTED SUCCESSFULLY!")

for file in output_folder.glob("*.csv"):
    print(file.name)


CLEANED DATASETS EXPORTED SUCCESSFULLY!
category_sales_items.csv
delivery_reviews.csv
orders_clean.csv
products_clean.csv


In [51]:

# STEP 80: CHECK SQLITE CONNECTION

import sqlite3

# Create a SQLite database inside our SQL folder
from pathlib import Path

sql_folder = Path("../sql")
sql_folder.mkdir(parents=True, exist_ok=True)

database_path = sql_folder / "ecommerce_analytics.db"

conn = sqlite3.connect(database_path)

print("SQLite database connected successfully!")
print("Database location:", database_path)

print("SQLite version:", sqlite3.sqlite_version)



SQLite database connected successfully!
Database location: ..\sql\ecommerce_analytics.db
SQLite version: 3.50.4


In [52]:

# STEP 81: LOAD CLEANED DATASETS INTO SQLITE

import pandas as pd
from pathlib import Path

# Location of cleaned CSV files
cleaned_folder = Path("../Data/cleaned")

# Tables to create in SQLite
tables = {
    "orders": "orders_clean.csv",
    "products": "products_clean.csv",
    "category_sales": "category_sales_items.csv",
    "delivery_reviews": "delivery_reviews.csv"
}

# Load each CSV into the database
for table_name, filename in tables.items():

    df = pd.read_csv(cleaned_folder / filename)

    df.to_sql(
        table_name,
        conn,
        if_exists="replace",
        index=False
    )

    print(f"{table_name}: {len(df):,} rows loaded")

# Confirm the database tables
print("\nDATABASE TABLES:")

tables_check = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn
)

display(tables_check)

print("\nAll cleaned datasets loaded successfully!")


orders: 99,441 rows loaded
products: 32,951 rows loaded
category_sales: 110,197 rows loaded
delivery_reviews: 96,470 rows loaded

DATABASE TABLES:


,name
0,category_sales
1,delivery_reviews
2,orders
3,products



All cleaned datasets loaded successfully!


In [53]:

# STEP 82: TOP 5 PRODUCT CATEGORIES USING SQL

query = """
SELECT
    category_english,
    COUNT(*) AS total_items,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(price), 2) AS merchandise_value,
    ROUND(AVG(price), 2) AS avg_item_price
FROM category_sales
GROUP BY category_english
ORDER BY merchandise_value DESC
LIMIT 5;
"""

top_categories_sql = pd.read_sql_query(query, conn)

print("TOP 5 PRODUCT CATEGORIES BY MERCHANDISE VALUE")

display(top_categories_sql)


TOP 5 PRODUCT CATEGORIES BY MERCHANDISE VALUE


,category_english,total_items,total_orders,merchandise_value,avg_item_price
0,health_beauty,9465,8647,1233131.72,130.28
1,watches_gifts,5859,5495,1166176.98,199.04
2,bed_bath_table,10953,9272,1023434.76,93.44
3,sports_leisure,8431,7530,954852.55,113.25
4,computers_accessories,7644,6530,888724.61,116.26


In [54]:

# STEP 83: DELIVERY PERFORMANCE AND CUSTOMER SATISFACTION

query = """
SELECT
    CASE
        WHEN late_delivery = 1 THEN 'Late'
        ELSE 'On Time'
    END AS delivery_status,

    COUNT(*) AS total_orders,

    ROUND(AVG(avg_review_score), 2) AS avg_review_score,

    SUM(
        CASE
            WHEN avg_review_score <= 2 THEN 1
            ELSE 0
        END
    ) AS low_rated_orders,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN avg_review_score <= 2 THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS low_rating_pct

FROM delivery_reviews

WHERE avg_review_score IS NOT NULL

GROUP BY
    CASE
        WHEN late_delivery = 1 THEN 'Late'
        ELSE 'On Time'
    END

ORDER BY delivery_status;
"""

delivery_sql = pd.read_sql_query(query, conn)

print("SQL DELIVERY AND CUSTOMER SATISFACTION ANALYSIS")

display(delivery_sql)


SQL DELIVERY AND CUSTOMER SATISFACTION ANALYSIS


,delivery_status,total_orders,avg_review_score,low_rated_orders,low_rating_pct
0,Late,6381,2.27,3979,62.36
1,On Time,89443,4.29,8257,9.23
